In [ ]:
import maria
import h5py
import arrow
import logging

import argparse
import numpy as np

import scipy as sp
import pandas as pd
from tqdm import tqdm

import jax
import jax.numpy as jnp
import jax.scipy as jsp

from maria.functions import normalized_matern
from scipy.stats import qmc
from functools import partial

import matplotlib.pyplot as plt
import matplotlib as mpl


In [ ]:
from maria import Quantity

seconds_per_year = 365 * 86400

default_parameters = {
    # "label": "default",
    "primary_size": Quantity(10, "m"),
    "T_eff": Quantity(5 * 0.2, "years"), # a five-year survey with 20% observing efficiency
    "f_sky": 0.5,
    "field_of_view": Quantity(3, "degrees"),
    "C_ell_white": Quantity(8e-8, "uK_CMB^2"),
    "scan_speed": Quantity(1, "deg/s"),
    "nu": Quantity(150, "GHz"),
    "z_max": Quantity(20000, "m"),
    "z_res": Quantity(10, "m"),
    "outer_scale": Quantity(500, "m"),
    "wind_scale": Quantity(0.5, "deg/s"),
    "scale_height": Quantity(1500, "m"),
    "pwv_rms": Quantity(2e-2, "mm"),
    "elevation": Quantity(45, "deg"),
}

ell = k = np.geomspace(10, 30000, 64)

# we want the total array NET to recover the forecasted survey NET; for HD it is
# Cl = 2.35 * 10^-8 uK^2
default_parameters["NET_array"] = (default_parameters["C_ell_white"] \
                                 * default_parameters["T_eff"] \
                                 / default_parameters["f_sky"] \
                                 / 16) ** 0.5

default_parameters

In [ ]:
experiment_name = "vary_primary_size"
surveys_list = [
        {"primary_size": Quantity(1, "m"), "label": r"$D = 1\,$m"},
        {"primary_size": Quantity(3, "m"), "label": r"$D = 3\,$m"},
        {"primary_size": Quantity(10, "m"), "label": r"$D = 10\,$m"},
        {"primary_size": Quantity(30, "m"), "label": r"$D = 30\,$m"},
        # {"primary_size": Quantity(100, "m"), "label": r"$100\,$m"}
          ]

In [ ]:
experiment_name = "vary_field_of_view"
surveys_list = [
        {"field_of_view": Quantity(0.3, "deg"), "label": r"$\theta_\text{fov} = 0.3^\circ$"},
        {"field_of_view": Quantity(1, "deg"), "label": r"$\theta_\text{fov} = 1^\circ$"},
        {"field_of_view": Quantity(3, "deg"), "label": r"$\theta_\text{fov} = 3^\circ$"},
        # {"field_of_view": Quantity(10, "deg"), "label": r"$\theta_\text{fov} = 10^\circ$"},
        # {"primary_size": Quantity(100, "m"), "label": r"$100\,$m"}
          ]

In [ ]:
experiment_name = "vary_scale_height"
surveys_list = [
        {"scale_height": Quantity(500, "m"), "label": r"$h_0 = 500\,$m"},
        {"scale_height": Quantity(1000, "m"), "label": r"$h_0 = 1\,$km"},
        {"scale_height": Quantity(1500, "m"), "label": r"$h_0 = 1.5\,$km"},
        {"scale_height": Quantity(2000, "m"), "label": r"$h_0 = 2\,$km"},
        # {"primary_size": Quantity(100, "m"), "label": r"$100\,$m"}
          ]

In [ ]:
experiment_name = "vary_outer_scale"
surveys_list = [
        {"outer_scale": Quantity(125, "m"), "label": r"$r_0 = 125\,$m"},
        {"outer_scale": Quantity(250, "m"), "label": r"$r_0 = 250\,$m"},
        {"outer_scale": Quantity(500, "m"), "label": r"$r_0 = 500\,$m"},
        {"outer_scale": Quantity(1000, "m"), "label": r"$r_0 = 1\,$km"},
        # {"primary_size": Quantity(100, "m"), "label": r"$100\,$m"}
          ]

In [ ]:
experiment_name = "vary_elevation"
surveys_list = [
        {"elevation": Quantity(60, "deg"), "label": r"$\epsilon = 60\circ$"},
        {"elevation": Quantity(50, "deg"), "label": r"$\epsilon = 50\circ$"},
        {"elevation": Quantity(45, "deg"), "label": r"$\epsilon = 45\circ$"},
        {"elevation": Quantity(40, "deg"), "label": r"$\epsilon = 40\circ$"},
        # {"primary_size": Quantity(100, "m"), "label": r"$100\,$m"}
          ]

In [ ]:


survey_entries_list = []
for survey in surveys_list:
    
    survey_entry = default_parameters.copy() # {k: survey.get(k, default_parameters[k]) for k in default_parameters}
    survey_entry.update(**survey)

    survey_entry["map_area"] = Quantity(4 * np.pi, "sr") * survey_entry["f_sky"]
    survey_entry["R_map"] = (survey_entry["map_area"] / np.pi) ** 0.5
        
    survey_entry["R_fov"] = survey_entry["field_of_view"] / 2
    # survey_entry["b"] = survey_entry["primary_size"] / (np.sqrt(8 * np.log(2)) * 2)

    survey_entry["fiducial"] = all([survey_entry[k] == default_parameters[k] for k in default_parameters])
        
    survey_entries_list.append(survey_entry)

surveys = pd.DataFrame(survey_entries_list)
surveys

In [ ]:


def abs_hum_rms_profile(z, base_abs_hum_rms, scale_height):
    return base_abs_hum_rms * np.exp(-z / scale_height)

# z_bins = np.geomspace(1e0, 2e4, 2000)
z_bins = np.arange(1e1, 2e4, 1e1)
dz = np.diff(z_bins)
z = (z_bins[1:] + z_bins[:-1]) / 2
zi, zj = np.meshgrid(z, z)
zdist = np.abs(zi - zj)

for survey_index, survey in surveys.iterrows():

    # an unnormalized profile with base_abs_hum_rms = 1 kg/m^3
    dummy_profile = abs_hum_rms_profile(z, 1, survey.scale_height.m)

    # this has units of kg^2/m^6
    dummy_turb_cov_mat = np.outer(dummy_profile, dummy_profile) * normalized_matern(zdist / survey.outer_scale.m, nu=1/3)

    # this has units of kg^2/m^4 = (kg/m^2)^2 = (mm)^2
    dummy_variance = np.sum(np.outer(dz, dz) * dummy_turb_cov_mat) # this has units of kg/m

    # so we renormalize by the desired total PWV RMS (in mm)
    surveys.loc[survey_index, "base_abs_hum_rms"] = survey.pwv_rms.mm / np.sqrt(dummy_variance)

    # print(f"desired pwv_rms = {survey.pwv_rms} | recovered pwv_rms = {Quantity(np.sqrt(np.sum(np.outer(dz, dz) * turb_cov_mat)), "m")}")

In [ ]:
from tqdm import trange

cmb = pd.read_csv("cmb_lensed.txt", sep=r"\s+")

g = {
    150: Quantity(9.59 / np.sqrt(2), "K_CMB/mm")
}

Cl_atm = {}
map_norm = {}

for survey_index, survey in surveys.iterrows():

    sigma_b = survey.primary_size / 4
    z_R = 4 * np.pi * sigma_b.m**2 * (survey.nu.Hz / 3e8)

    f_sky = survey.R_map**2 / 4
    numeric_part = 128 * np.sqrt(2) * np.pi**2 * sp.special.gamma(11/6) / sp.special.gamma(1/3)
    survey_part = f_sky / (survey.R_fov**2 * survey.T_eff)
    atm_part = g[survey.nu.GHz]**2 / (survey.wind_scale * survey.outer_scale ** (2/3))
    ell_part = k ** (-14/3)
    
    Cl_atm[survey_index] = 0.

    dz = Quantity(1e2, "m")
    z_samples = np.arange(dz.m, 20000, dz.m)

    for z_index in trange(len(z_samples), desc=survey.label):

        z = Quantity(z_samples[z_index], "m")
        sigma_b_z = sigma_b * np.sqrt(1 + (z.m / z_R)**2)

        # this is in units of (kg/m^3)^2 = (mm_pwv/m)^2, so we need to scale by 1e-6
        scale_part = 1e-6 * abs_hum_rms_profile(z,
                                                survey.base_abs_hum_rms, 
                                                survey.scale_height / np.sin(survey.elevation.rad)) ** 2
        
        beam_part = np.exp(-(sigma_b_z * k / z)**2)
        z_part = z**(5/3)

        Cl_atm[survey_index] = Cl_atm[survey_index] + dz * z_part * scale_part * beam_part

    Cl_atm[survey_index] *= numeric_part * survey_part * atm_part * ell_part * Quantity(1, "rad")
    Cl_atm[survey_index] = sp.interpolate.interp1d(ell, 
                                                   Cl_atm[survey_index].to("uK_CMB^2"), 
                                                   kind="cubic", bounds_error=False)(cmb.ells.values)
    

In [ ]:
fontsize = 10

fig, ax = plt.subplots(1, 1, figsize=(4.5, 4.5), constrained_layout=True, dpi=256)

def latex_scientific_notation_repr(x):

    if not np.size(x) == 1:
        raise ValueError()
    
    power = np.floor(np.log10(x))    
    return fr"{x * 10 **-power:.0f} \times 10^{{{int(power)}}}"

for index, entry in tqdm(surveys.iterrows()):

    ax.plot(cmb.ells.values, Cl_atm[index], lw=1, label=entry.label)

fid_index = surveys.index.values[list(surveys.fiducial.values).index(True)]
pl = Cl_atm[fid_index][100] * (cmb.ells.values / cmb.ells.values[100]) ** (-14/3)
ax.plot(cmb.ells, pl, lw=1e0, ls=":", c="k", label=r"$\ell^{-14/3}$")

ax.plot(cmb.ells, cmb.tt, lw=1e0, label="CMB", color="k")

Cl_white = entry.C_ell_white.to("uK_CMB^2")
ax.plot(
    [cmb.ells.min(), cmb.ells.max()],
    [Cl_white, Cl_white],
    c="k",
    lw=2,
    alpha=0.4,
)


ax.annotate(text=fr"$C_{{\ell,\text{{white}}}} = {latex_scientific_notation_repr(Cl_white)}$ uK$_\text{{CMB}}^2$",
            xy=(60, 1.2 * Cl_white), 
            fontsize=fontsize,
            ha="left", 
            va="bottom")

ax.plot()

ax.set_xlabel(r"$\ell$")
ax.loglog()
ax.set_xlim(5e1, 2e4)

ax.legend(fontsize=fontsize)
ax.set_ylabel(r"$C_\ell$ [uK$_\text{CMB}^2$]")
ax.set_ylim(1e-8, 1e2)

plt.savefig(f"N_ell_{experiment_name}.pdf", dpi=256)